# QFI Scaling With System Size

This notebook plots how directional QFI scales with system size for several initial-state families. It uses the dataset generated by `python run_qfi_scaling.py`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

DATA_DIR = Path("data/qfi_scaling")
plt.rcParams.update({"figure.dpi": 140})

def load_config(dataset_dir):
    with open(dataset_dir / "config.json") as f:
        return json.load(f)

def choose_dataset(data_dir, experiment_name="qfi_scaling"):
    candidates = []
    for dataset_dir in sorted(data_dir.iterdir()) if data_dir.exists() else []:
        if not (dataset_dir / "config.json").exists() or not (dataset_dir / "results.npz").exists():
            continue
        config = load_config(dataset_dir)
        if config.get("experiment_name") != experiment_name:
            continue
        candidates.append(("__smoke" in dataset_dir.name, dataset_dir.stat().st_mtime, dataset_dir.name, dataset_dir, config))
    if not candidates:
        raise FileNotFoundError(f"No {experiment_name} dataset found in {data_dir}. Run python run_qfi_scaling.py first.")
    candidates.sort(key=lambda item: (item[0], -item[1]))
    _, _, name, dataset_dir, config = candidates[0]
    results = np.load(dataset_dir / "results.npz", allow_pickle=False)
    return name, config, results

In [ ]:
scaling_dataset, config, data = choose_dataset(DATA_DIR)

n_values = data["n_values"]
state_labels = data["state_labels"]
qfi_values = data["qfi"]
exponents = data["scaling_exponents"]

print("dataset:", scaling_dataset)
print(config["experiment_name"])
print(config["scan"]["description"])
print("n values:", n_values)
print("states:", state_labels)
print("exponents:", exponents)

## Directional QFI Scaling

A slope near 1 indicates extensive scaling; a slope near 2 indicates Heisenberg-like scaling.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for label, qfi in zip(state_labels, qfi_values):
    ax.loglog(n_values, qfi, marker="o", label=str(label))

ax.xaxis.set_major_locator(mticker.FixedLocator(n_values))
ax.xaxis.set_major_formatter(mticker.FixedFormatter([str(n) for n in n_values]))
ax.set_xlabel(r"system size $n$")
ax.set_ylabel(r"$v^T \mathcal{F}_Q v$")
ax.set_title("Directional QFI scaling")
ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.3))
ax.bar([str(label) for label in state_labels], exponents)
ax.axhline(1.0, color="black", lw=1, ls="--", label=r"$n$")
ax.axhline(2.0, color="gray", lw=1, ls=":", label=r"$n^2$")
ax.set_ylabel("log-log slope")
ax.set_title("Estimated QFI scaling exponent")
ax.tick_params(axis="x", rotation=20)
ax.legend(frameon=False)
fig.tight_layout()